### 차트 작성 시 한글 깨짐 방지를 위한 koreanize-matplotlib 설치

In [1]:
!pip install koreanize-matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 23.2 MB/s eta 0:00:00


### 라이브러리 import

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

### 1. 데이터 로드 및 확인

In [3]:
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target
feature_names = diabetes.feature_names

In [4]:
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y
display(df.head())

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


### 2. 학습 데이터 분할

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((353, 10), (89, 10), (353,), (89,))

### 3. 선형 회귀 모델 학습

In [6]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

### 4. 예측

In [10]:
y_pred = lr.predict(X_test)

### 5. 오차 기반 지표(MAE, MSE, RMSE)

In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error, mean_squared_log_error

# 1. MAE (Mean Absolute Error): 오차의 절대값 평균
# 특징: 직관적이며, 이상치에 덜 민감함
mae = mean_absolute_error(y_test, y_pred)

# 2. MSE (Mean Squared Error): 오차 제곱의 평균
# 특징: 이상치(큰 오차)에 민감하게 반응하여 페널티를 부여
mse = mean_squared_error(y_test, y_pred)

# 3. RMSE (Root Mean Squared Error): MSE의 제곱근
# 특징: MSE의 단위를 다시 원래 데이터 단위로 맞춤. 큰 오차에 민감함.
rmse = np.sqrt(mse)

print(f"1. MAE  (평균 절대 오차)   : {mae:.4f}")
print(f"2. MSE  (평균 제곱 오차)   : {mse:.4f}")
print(f"3. RMSE (평균 제곱근 오차) : {rmse:.4f}")

1. MAE  (평균 절대 오차)   : 42.7941
2. MSE  (평균 제곱 오차)   : 2900.1936
3. RMSE (평균 제곱근 오차) : 53.8534


### 6. 비율 및 로그 기반 지표 (MAPE, RMSLE)

In [13]:
# 4. MAPE (Mean Absolute Percentage Error): 평균 절대 백분율 오차
# 특징: 오차를 퍼센트(%)로 표현하여 스케일이 다른 모델과 비교 용이
mape = mean_absolute_percentage_error(y_test, y_pred)

# 5. RMSLE (Root Mean Squared Log Error): 평균 제곱근 로그 오차
# 특징: 과소예측(Underestimation)에 더 큰 페널티, 값의 범위가 클 때 유용
# (주의: 음수가 있을 경우 오류 발생 가능하므로 보통 np.log1p 사용, 여기서는 sklearn 함수 사용)
# 데이터에 음수가 없어야 하므로 확인
if (y_test < 0).any() or (y_pred < 0).any():
    print("음수 값이 있어 RMSLE 계산 시 주의 필요 (값을 보정합니다)")
    # 음수 예측값이 나올 경우 0으로 보정 (RMSLE 계산을 위해)
    y_pred_safe = np.maximum(y_pred, 0)
    rmsle = np.sqrt(mean_squared_log_error(y_test, y_pred_safe))
else:
    rmsle = np.sqrt(mean_squared_log_error(y_test, y_pred))

print(f"4. MAPE  (평균 절대 비율 오차): {mape:.4f} ({mape*100:.2f}%)")
print(f"5. RMSLE (평균 제곱근 로그 오차): {rmsle:.4f}")

4. MAPE  (평균 절대 비율 오차): 0.3750 (37.50%)
5. RMSLE (평균 제곱근 로그 오차): 0.4207


### 7. 모델 설명력 평가 (R2, Adjusted R2)

In [17]:
# 6. R2 Score (결정 계수)
# 특징: 데이터의 분산을 모델이 얼마나 설명하는지 (1에 가까울수록 좋음)
r2 = r2_score(y_test, y_pred)

# 7. Adjusted R2 Score (수정된 결정 계수)
# 특징: 독립 변수의 개수가 늘어날 때 발생하는 R2 스코어의 증가를 보정 (회귀 분석 모델 간 비교에 유용)

n = len(y_train)  # 학습 데이터의 관측치 수
p = X_train.shape[1]  # 독립 변수의 수

adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"6. R-squared (Scikit-Learn) : {r2:.4f}")
print(f"7. Adjusted R-squared (Formula): {adjusted_r2:.4f}")


6. R-squared (Scikit-Learn) : 0.4526
7. Adjusted R-squared (Formula): 0.4366
